# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
#import 
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import gradio as gr
import requests

# imports to handle image generation
import base64
from io import BytesIO
from PIL import Image

# Initialization
load_dotenv(override=True)
open_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if open_api_key:
    print(f"OK - OpenAI API Key exists and begins {open_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"OK - Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:  
    print(f"OK - Google API Key exists and begins {google_api_key[:8]}")    
else:
    print("Google API Key not set")


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI()
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)   
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key='ollama', base_url=ollama_url)


# Models
model_gpt = "gpt-4o-mini"
model_claude = "claude-sonnet-4-5-20250929"
model_gemini = "gemini-2.5-flash-lite"
model_ollama = "gpt-oss:20b"

#print "Ollama is running" , otherwise open a terminal and run `ollama serve`
# to make it work in the notebook, it must be the last line in the cell, otherwise it run silently
requests.get("http://localhost:11434").content 
# instead, use this line to check if ollama is running
print("Is Ollama running?  : ", requests.get("http://localhost:11434").content)

In [ ]:
#prompt for system
system_message = """
You are a helpful assistant that can answer the question
provided by the user in markdown format without code blocks.
Make the answer concise and to the point.
Keep the answer strictly related to the topic and within 50 words maximum - No exceptions.
Get the single word reference for the image on that topic which can be used for image generation strictly on that topic.
"""
MODEL_CONFIG = {
    "gpt": (openai, model_gpt),
    "claude": (anthropic, model_claude),
    "gemini": (gemini, model_gemini),
    "ollama": (ollama, model_ollama),
}


In [ ]:
# stream response from the LLM - Try 1
def stream_response(prompt, model_key):
    if model_key not in MODEL_CONFIG:
        raise ValueError(f"Unknown model_key: {model_key}")
    client, model_name = MODEL_CONFIG[model_key]
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]
    stream = client.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True,
    )
    result = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        result += delta
        yield result

In [ ]:
message_input = gr.Textbox(label="As a Question message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["gpt", "claude","gemini","ollama"], label="Select model", value="gpt")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_response,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "gpt"],
            ["What depicts on Utah state flag", "claude"]
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
#prompt for system
system_message = """
You are a helpful assistant that can answer the question
provided by the user in markdown format without code blocks.
Make the answer concise and to the point.
Keep the answer strictly related to the topic and within 50 words maximum - No exceptions.
Get the single word reference for the image on that topic which can be used for image generation strictly on that topic - Can use underscore to identify better image.
"""
message_input = gr.Textbox(label="As a Question - try one liner:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["gpt", "claude","gemini","ollama"], label="Select model", value="gpt")
message_output = gr.Markdown(label="Response:")

import json
import re

def artist(keyword):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing the {keyword}, showing most common image related to {keyword}, in a real world like style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

def parse_llm_payload(raw: str):
    raw = raw.strip()
    try:
        data = json.loads(raw)
        answer = data["answer"].strip()
        keyword = (data.get("image_keyword") or "").strip()
        return answer, keyword
    except json.JSONDecodeError:
        pass
    m = re.search(r'(?im)\breference:\s*"([^"]+)"', raw)
    if not m:
        m = re.search(r"(?im)\breference:\s*([^\s.]+)", raw)
    if m:
        keyword = m.group(1).strip()
        answer = re.sub(r'(?im)[\s\n]*reference:\s*"?[^"\n]+"?[\s.]*$', "", raw).strip()
        return answer, keyword
    # IMAGE_KEYWORD line (optional)
    m = re.search(r"(?im)^IMAGE_KEYWORD:\s*(.+)\s*$", raw)
    if m:
        keyword = m.group(1).strip()
        answer = re.sub(r"(?im)^IMAGE_KEYWORD:\s*.+\s*$", "", raw).strip()
        return answer, keyword
    return raw, ""   # <-- required: no keyword extracted

def respond_with_image(prompt, model_key):
    client, model_name = MODEL_CONFIG[model_key]
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_message},  # must require JSON or IMAGE_KEYWORD format
            {"role": "user", "content": prompt},
        ],
        stream=False,
    )
    raw = completion.choices[0].message.content or ""
    answer_text, keyword = parse_llm_payload(raw)
    if keyword:
        pil_image = artist(keyword)
    else:
        pil_image = None  # or a placeholder; None clears/hides depending on Gradio version
    return answer_text, pil_image
# Gradio Interface with image generation

gr.Interface(
    fn=respond_with_image,
    inputs=[message_input, model_selector],
    outputs=[gr.Markdown(label="Response"), 
    gr.Image(label="Illustration")],
    flagging_mode="never").launch(share=True)